Deze code kan je doorlopen om van de nc bestanden van biooracle een dataset te maken met de juiste locaties zoals onze oorspronkelijke dataset

In [1]:
import pandas as pd
import xarray as xr
import glob
import numpy as np #

In [2]:
df1 = pd.read_csv("Data/Diversity_data_with_biooracle_2010.csv")
df1.columns
df1.head()

,marine_species_richness,PD,co1_genetic_diversity_mean,long_deg,lat_deg,chl_max,chl_mean,chl_min,clt_max,clt_mean,...,terrain_characteristics_bea_min,terrain_characteristics_rug,terrain_characteristics_slope,terrain_characteristics_topo,T_ltmax,T_ltmin,T_max,T_mean,T_min,T_range
0,44,11.830844,0.007171,-171.74579,-71.23536,1.987486,0.264222,0.015311,0.993542,0.681597,...,-3883.0,66.347260,0.933108,6.854279,-0.711044,-1.904281,-0.127859,-1.602750,-1.977225,1.849366
1,43,11.830844,0.007171,-167.88679,-71.23536,2.205368,0.278952,0.015232,0.961121,0.693978,...,-4084.0,21.055542,0.320794,-12.930603,-0.847708,-1.925544,-0.017505,-1.620118,-2.000000,1.988636
2,44,12.650381,0.007499,-164.02779,-71.23536,1.891105,0.270836,0.015052,0.960210,0.698352,...,-4153.0,14.489563,0.228825,5.524109,-0.521772,-1.923338,0.125601,-1.582915,-2.000000,2.137620
3,44,12.650381,0.007499,-160.16879,-71.23536,1.963815,0.293940,0.015424,0.962838,0.693123,...,-4303.0,4.413208,0.081748,1.760254,-0.558407,-1.909406,-0.032617,-1.579760,-1.976150,1.943533
4,45,12.650381,0.007499,-156.30979,-71.23536,1.956123,0.298378,0.015726,0.969080,0.688964,...,-4341.0,10.302124,0.234018,-0.892456,-0.570859,-1.906482,-0.000029,-1.570748,-2.000000,2.001978


maak een dataset aan met enkel de locaties (dit hebben we straks nodig)

In [3]:
df_locations = df1[["long_deg", "lat_deg"]]
df_locations.head(10)

,long_deg,lat_deg
0,-171.74579,-71.23536
1,-167.88679,-71.23536
2,-164.02779,-71.23536
3,-160.16879,-71.23536
4,-156.30979,-71.23536
5,-152.45079,-71.23536
6,-148.59179,-71.23536
7,-144.73279,-71.23536
8,-140.87379,-71.23536
9,-59.83479,-71.23536


In [4]:
df2 = pd.read_csv("Data/Diversity_data_with_env.csv")
df2.head()

,grid_id,long,lat,marine_species_richness,PD,co1_genetic_diversity_mean,long_deg,lat_deg,long_dd,lat_dd,...,Shelf,Slope,Abyssal,TidalRange,Coral,Estuary,Seamount,MPA,matched_CenterLong,matched_CenterLat
0,1,-17174579,-7123536,44,11.830844,0.007171,-171.74579,-71.23536,-171.74579,-71.23536,...,0.0,730.228795,263.361205,-9999.0,0.0,0.0,0,NaN,-171.75,-71.25
1,2,-16788679,-7123536,43,11.830844,0.007171,-167.88679,-71.23536,-167.88679,-71.23536,...,0.0,0.000000,993.590000,-9999.0,0.0,0.0,0,NaN,-167.75,-71.25
2,3,-16402779,-7123536,44,12.650381,0.007499,-164.02779,-71.23536,-164.02779,-71.23536,...,0.0,0.000000,993.590000,-9999.0,0.0,0.0,0,NaN,-164.25,-71.25
3,4,-16016879,-7123536,44,12.650381,0.007499,-160.16879,-71.23536,-160.16879,-71.23536,...,0.0,0.000000,993.590000,-9999.0,0.0,0.0,0,NaN,-160.25,-71.25
4,5,-15630979,-7123536,45,12.650381,0.007499,-156.30979,-71.23536,-156.30979,-71.23536,...,0.0,0.000000,993.590000,-9999.0,0.0,0.0,0,NaN,-156.25,-71.25


Verzamel al je nc bestanden in een map zoals de map Data.
Ik zou per biodiversiteitsmaat een andere naam aan de map geven (mss ncPD, ncGD en ncSR?)

In [5]:
files = glob.glob("ncPD/*.nc")

Van alle aparte nc bestandjes 1 geheeld maken 

In [6]:
datasets = [xr.open_dataset(f) for f in files]
ds = xr.merge(datasets)

print(ds)

<xarray.Dataset> Size: 0B
Dimensions:  ()
Data variables:
    *empty*


De dataset omvormen tot een dataframe waarmee je kan verder werken om aanpassingen aan te brengen 

In [ ]:
df = ds.to_dataframe().reset_index()

In [ ]:
len(df)  # is nog veel te lang dus veel locaties moeten nog wegvallen => komt straks

In [ ]:
df.head()    # je ziet dat longitude en latitude de variabelen zijn maar we hebben long_deg en lat_deg nodig

In [ ]:
df = df.rename(columns={"longitude": "long_deg", "latitude": "lat_deg"})

Je moet long_deg en lat_deg afronden tot op 5 cijfers na de komma anders zit je met andere komma getallen dan bij de oorspronkelijke dataset

In [ ]:
print(df[["lat_deg", "long_deg"]].dtypes)
df["lat_deg"] = df["lat_deg"].astype("float64").round(5)
df["long_deg"] = df["long_deg"].astype("float64").round(5)
print(df[["lat_deg", "long_deg"]].head())

De volgende code is om de locaties te laten kloppen zoals in de oorspronkelijke dataset

In [ ]:
# functie om dichtstbijzijnde waarde te vinden
def nearest(array, values):
    """
    array: 1D array van rasterwaarden (lat of long)
    values: array van gewenste waarden
    return: voor elke waarde in values de dichtstbijzijnde in array
    """
    array = np.array(array)
    values = np.array(values)
    idx = np.abs(array[:, None] - values).argmin(axis=0)
    return array[idx]

# vind dichtstbijzijnde lat/lon
df_locations["lat_match"] = nearest(df["lat_deg"].unique(), df_locations["lat_deg"])
df_locations["long_match"] = nearest(df["long_deg"].unique(), df_locations["long_deg"])

# merge op de “nearest” kolommen
df_subset = df.merge(
    df_locations,
    left_on=["lat_deg", "long_deg"],
    right_on=["lat_match", "long_match"],
    how="inner"
)

In [ ]:
df_subset.head()  #je ziet dat long_deg_y en lat_deg_y exact overeenkomen met de originele dataset => al de rest
# van long en lat variabelen kan je verwijderen 

In [ ]:
df_subset = df_subset.drop(["lat_match","long_match", "long_deg_x", "lat_deg_x", "time"], axis = 1)
df_subset.head()

In [ ]:
len(df_subset)  # lengte komt exact overeen met oorspronkelijke dataset

In [ ]:
df_subset = df_subset.rename(columns={"long_deg_y": "long_deg", "lat_deg_y": "lat_deg"})
df_subset.head()

De volgende 2 stukjes code moeten true terug geven (dit is ter controle) wel belangrijk!!!!

In [ ]:
(df1['lat_deg'] == df_subset['lat_deg']).all()  # True als alle rijen exact overeenkomen

In [ ]:
(df1['long_deg'] == df_subset['long_deg']).all() # True als alle rijden exact overeenkomen

De volgende code is afhankelijk van welke variabelen je in je model steekt (dus niet exact kopiëren)

In [ ]:
df_subset = df_subset.rename(columns={
    "so_mean": "salinity_mean",
    "swd_mean": "currentdirection_mean",
    "sws_mean": "currentvelocity_mean",
    "thetao_mean": "T_mean"
})
df_subset.columns

In [ ]:
df_subset.head()

In [ ]:
df2 = pd.read_csv("Data/Diversity_data_with_env.csv")
df2.head()

Nog de variabelen van env toevoegen die je ook nodig hebt voor in je model 

In [ ]:
df_merged = df_subset.merge(
    df2[['long_deg', 'lat_deg', 'DepthMean', 'LandDist', 'Slope', 'Abyssal']],
    on=['long_deg', 'lat_deg'],
    how='left'
)
df_merged.head()

Sla je dataset op als csv bestand dan kan je dit gebruiken in je ander documentje met je model 

In [ ]:
df_merged.to_csv("PD_2050-60_SSP119.csv")